<a href="https://colab.research.google.com/github/N-Uma/Machine-Learning-and-Big-Data/blob/main/Model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from pyspark.sql import SparkSession
from pyspark.ml.regression import (
    LinearRegression,
    GBTRegressor,
    RandomForestRegressor
)
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
import pyspark.sql.functions as F



BASE_PATH = "/content/drive/MyDrive/NYC_Taxi_Yellow_2012"


spark = SparkSession.builder.getOrCreate()
print(f"Spark {spark.version} ready")

train_df = spark.read.parquet(f"{BASE_PATH}/data/final/train_2012")
val_df   = spark.read.parquet(f"{BASE_PATH}/data/final/val_2012")
test_df  = spark.read.parquet(f"{BASE_PATH}/data/final/test_2012")

train_count = train_df.count()
val_count   = val_df.count()
test_count  = test_df.count()

print(f"Train: {train_count:,} rows | Schema: {train_df.columns}")
print(f"Val:   {val_count:,} rows")
print(f"Test:  {test_count:,} rows")

train_df.show(3, truncate=False)

Spark 4.0.2 ready
Train: 140,517,318 rows | Schema: ['features', 'fare_amount']
Val:   13,663,801 rows
Test:  14,572,953 rows
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+
|features                                                                                                                                                                                                                                                               |fare_amount|
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+
|[0.23388725290758766,0.9758800005496988

In [5]:
from pyspark.ml.regression import (
    LinearRegression,
    GBTRegressor,
    RandomForestRegressor
)

models_config = {
    "1. Linear Regression": LinearRegression(
        featuresCol="features",
        labelCol="fare_amount",
        maxIter=50,
        regParam=0.1,
        elasticNetParam=0.5
    ),

    "2. Gradient Boosted Trees": GBTRegressor(
        featuresCol="features",
        labelCol="fare_amount",
        maxIter=20,
        maxDepth=6,
        stepSize=0.05
    ),

    "3. Random Forest": RandomForestRegressor(
        featuresCol="features",
        labelCol="fare_amount",
        numTrees=20,
        maxDepth=8,
        featureSubsetStrategy="sqrt"
    ),

    "4. Ridge Regression": LinearRegression(
        featuresCol="features",
        labelCol="fare_amount",
        maxIter=50,
        regParam=1.0,
        elasticNetParam=0.0
    )
}


print("Configured MLlib Regression Models:")
for model_name in models_config:
    print(f"  • {model_name}")

Configured MLlib Regression Models:
  • 1. Linear Regression
  • 2. Gradient Boosted Trees
  • 3. Random Forest
  • 4. Ridge Regression


In [6]:
from pyspark.ml.evaluation import RegressionEvaluator


train_small = (
    train_df
    .select("features", "fare_amount")
    .sample(
        withReplacement=False,
        fraction=0.01,
        seed=42
    )
    .cache()
)

val_small = (
    val_df
    .select("features", "fare_amount")
    .sample(
        withReplacement=False,
        fraction=0.05,
        seed=42
    )
    .cache()
)


train_count = train_small.count()
val_count   = val_small.count()

print(f"Train sample: {train_count:,} rows")
print(f"Validation sample: {val_count:,} rows")


rmse_evaluator = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="rmse"
)

r2_evaluator = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="r2"
)

trained_models = {}
validation_results = []

for model_name, estimator in models_config.items():

    print(f"\nTraining {model_name}...")

    trained_model = estimator.fit(train_small)
    trained_models[model_name] = trained_model

    val_predictions = trained_model.transform(val_small)

    rmse = rmse_evaluator.evaluate(val_predictions)
    r2 = r2_evaluator.evaluate(val_predictions)

    validation_results.append(
        (model_name, __builtins__.round(rmse, 3), __builtins__.round(r2, 4))
    )

    print(f"RMSE: ${rmse:.2f} | R²: {r2:.4f}")

results_df = spark.createDataFrame(
    validation_results,
    ["Model", "RMSE", "R2"]
)

print("\nValidation Results (sorted by RMSE):")
results_df.orderBy("RMSE").show(truncate=False)

train_small.unpersist()
val_small.unpersist()

Train sample: 1,406,796 rows
Validation sample: 683,521 rows

Training 1. Linear Regression...
RMSE: $4.15 | R²: 0.8168

Training 2. Gradient Boosted Trees...
RMSE: $4.16 | R²: 0.8166

Training 3. Random Forest...
RMSE: $4.65 | R²: 0.7708

Training 4. Ridge Regression...
RMSE: $4.43 | R²: 0.7914

Validation Results (sorted by RMSE):
+-------------------------+-----+------+
|Model                    |RMSE |R2    |
+-------------------------+-----+------+
|1. Linear Regression     |4.154|0.8168|
|2. Gradient Boosted Trees|4.156|0.8166|
|4. Ridge Regression      |4.432|0.7914|
|3. Random Forest         |4.646|0.7708|
+-------------------------+-----+------+



DataFrame[features: vector, fare_amount: double]

In [7]:
from pyspark.ml.evaluation import RegressionEvaluator

test_data = test_df.select("features", "fare_amount")

rmse_evaluator = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="rmse"
)

mae_evaluator = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="mae"
)

r2_evaluator = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="r2"
)

print(f"{'Model':<25} {'Test RMSE ($)':<15} {'Test MAE ($)':<15} {'Test R²':<10}")

test_results = {}

for model_name, trained_model in trained_models.items():

    print(f"\nEvaluating on test set: {model_name}")

    test_predictions = trained_model.transform(test_data)

    rmse = rmse_evaluator.evaluate(test_predictions)
    mae = mae_evaluator.evaluate(test_predictions)
    r2 = r2_evaluator.evaluate(test_predictions)

    test_results[model_name] = {
        "Test_RMSE": __builtins__.round(rmse, 3),
        "Test_MAE": __builtins__.round(mae, 3),
        "Test_R2": __builtins__.round(r2, 4)
    }

    print(f"  RMSE: ${rmse:.2f} | MAE: ${mae:.2f} | R²: {r2:.4f}")

print("\nFINAL RANKING:")

sorted_results = sorted(
    test_results.items(),
    key=lambda item: item[1]["Test_RMSE"]
)

for rank, (model_name, scores) in enumerate(sorted_results, start=1):
    print(
        f"{rank}. {model_name:<23} "
        f"RMSE=${scores['Test_RMSE']:6.2f}  "
        f"MAE=${scores['Test_MAE']:6.2f}  "
        f"R²={scores['Test_R2']:7.4f}"
    )

best_model_name = sorted_results[0][0]
print(f"\nBEST MODEL: {best_model_name}")

Model                     Test RMSE ($)   Test MAE ($)    Test R²   

Evaluating on test set: 1. Linear Regression
  RMSE: $4.20 | MAE: $2.12 | R²: 0.8101

Evaluating on test set: 2. Gradient Boosted Trees
  RMSE: $4.22 | MAE: $2.04 | R²: 0.8088

Evaluating on test set: 3. Random Forest
  RMSE: $4.69 | MAE: $2.55 | R²: 0.7632

Evaluating on test set: 4. Ridge Regression
  RMSE: $4.48 | MAE: $2.37 | R²: 0.7844

FINAL RANKING:
1. 1. Linear Regression    RMSE=$  4.20  MAE=$  2.12  R²= 0.8101
2. 2. Gradient Boosted Trees RMSE=$  4.22  MAE=$  2.04  R²= 0.8088
3. 4. Ridge Regression     RMSE=$  4.48  MAE=$  2.37  R²= 0.7844
4. 3. Random Forest        RMSE=$  4.69  MAE=$  2.56  R²= 0.7632

BEST MODEL: 1. Linear Regression


In [8]:
from pyspark.ml.regression import (
    GBTRegressor,
    RandomForestRegressor,
    LinearRegression
)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

print(f"\nTUNING BEST MODEL: {best_model_name}")

if "Gradient Boosted Trees" in best_model_name:
    estimator = GBTRegressor(
        featuresCol="features",
        labelCol="fare_amount"
    )

    param_grid = (
        ParamGridBuilder()
        .addGrid(estimator.maxDepth, [3, 5])
        .addGrid(estimator.maxIter, [10, 20])
        .build()
    )

elif "Random Forest" in best_model_name:
    estimator = RandomForestRegressor(
        featuresCol="features",
        labelCol="fare_amount"
    )

    param_grid = (
        ParamGridBuilder()
        .addGrid(estimator.maxDepth, [4, 6])
        .addGrid(estimator.numTrees, [10, 20])
        .build()
    )

else:
    estimator = LinearRegression(
        featuresCol="features",
        labelCol="fare_amount"
    )

    param_grid = (
        ParamGridBuilder()
        .addGrid(estimator.regParam, [0.01, 0.1])
        .addGrid(estimator.maxIter, [50, 100])
        .build()
    )

train_cv_sample = (
    train_df
    .select("features", "fare_amount")
    .sample(
        withReplacement=False,
        fraction=0.01,
        seed=42
    )
    .cache()
)
train_cv_sample.count()

evaluator = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="rmse"
)

cv = CrossValidator(
    estimator=estimator,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=2
)

print("Running 2-fold cross-validation on sampled data...")

cv_model = cv.fit(train_cv_sample)

test_data = test_df.select("features", "fare_amount")
test_predictions = cv_model.bestModel.transform(test_data)

test_rmse = evaluator.evaluate(test_predictions)

print("\nBest Model Parameters:")
print(cv_model.bestModel.extractParamMap())

print(f"\nTuned Test RMSE: ${test_rmse:.2f}")

train_cv_sample.unpersist()


TUNING BEST MODEL: 1. Linear Regression
Running 2-fold cross-validation on sampled data...

Best Model Parameters:
{Param(parent='LinearRegression_80de71979b9f', name='aggregationDepth', doc='suggested depth for treeAggregate (>= 2).'): 2, Param(parent='LinearRegression_80de71979b9f', name='elasticNetParam', doc='the ElasticNet mixing parameter, in range [0, 1]. For alpha = 0, the penalty is an L2 penalty. For alpha = 1, it is an L1 penalty.'): 0.0, Param(parent='LinearRegression_80de71979b9f', name='epsilon', doc='The shape parameter to control the amount of robustness. Must be > 1.0. Only valid when loss is huber'): 1.35, Param(parent='LinearRegression_80de71979b9f', name='featuresCol', doc='features column name.'): 'features', Param(parent='LinearRegression_80de71979b9f', name='fitIntercept', doc='whether to fit an intercept term.'): True, Param(parent='LinearRegression_80de71979b9f', name='labelCol', doc='label column name.'): 'fare_amount', Param(parent='LinearRegression_80de7197

DataFrame[features: vector, fare_amount: double]

In [9]:
best_model = None
best_model_name = None

for model_name, trained_model in trained_models.items():
    if model_name == best_model_name or best_model is None:
        best_model = trained_model
        best_model_name = model_name

feature_names = [
    "passenger_count",
    "trip_distance",
    "haversine_km",
    "trip_duration_min",
    "pickup_hour",
    "is_peak_hour",
    "is_weekend",
    "dow_vector"
]

print(f"\nMODEL INTERPRETABILITY ({best_model_name})")
print(f"{'Feature':<22} {'Value'}")

if hasattr(best_model, "featureImportances"):

    importances = best_model.featureImportances.toArray()

    for feature, importance in zip(feature_names, importances):
        print(f"{feature:<22} {importance:.4f}")

elif hasattr(best_model, "coefficients"):

    coefficients = best_model.coefficients.toArray()

    for feature, coef in zip(feature_names, coefficients):
        print(f"{feature:<22} {coef:+.4f}")

    print(f"\nIntercept: {best_model.intercept:+.4f}")

else:
    print("Model does not support interpretability output.")


MODEL INTERPRETABILITY (1. Linear Regression)
Feature                Value
passenger_count        +0.0000
trip_distance          +7.6050
haversine_km           +0.0000
trip_duration_min      +0.0048
pickup_hour            +0.0594
is_peak_hour           -0.0699
is_weekend             +0.0000
dow_vector             +0.0000

Intercept: +10.6653


In [11]:
from pyspark.sql import functions as F

print("\nRESIDUAL ANALYSIS")

test_data = test_df.select("features", "fare_amount")

best_model = trained_models[best_model_name]

test_predictions = best_model.transform(test_data)

test_with_residuals = test_predictions.withColumn(
    "residual",
    F.col("fare_amount") - F.col("prediction")
)

test_with_residuals.select(
    "prediction",
    "fare_amount",
    "residual"
).show(10, truncate=False)

residual_stats = test_with_residuals.agg(
    F.round(F.avg("residual"), 2).alias("mean_residual"),
    F.round(F.stddev("residual"), 2).alias("residual_std")
)

print("\nResidual Summary Statistics:")
residual_stats.show()


RESIDUAL ANALYSIS
+-----------------+-----------+--------------------+
|prediction       |fare_amount|residual            |
+-----------------+-----------+--------------------+
|8.872051829593202|9.0        |0.12794817040679796 |
|5.97238252179999 |6.0        |0.027617478200009593|
|23.69874155338005|25.5       |1.8012584466199506  |
|9.09586805295712 |12.5       |3.4041319470428792  |
|6.323477104586477|8.0        |1.6765228954135232  |
|9.781536857597319|9.5        |-0.2815368575973185 |
|7.317718770885552|7.5        |0.182281229114448   |
|6.88260596447911 |6.5        |-0.3826059644791098 |
|9.117748460300474|11.0       |1.882251539699526   |
|7.782053664679072|9.5        |1.7179463353209279  |
+-----------------+-----------+--------------------+
only showing top 10 rows

Residual Summary Statistics:
+-------------+------------+
|mean_residual|residual_std|
+-------------+------------+
|         1.64|        3.87|
+-------------+------------+



In [10]:
from pyspark.sql import functions as F

print("\nSAVING MODELS & EXPORTING RESULTS")

safe_model_name = best_model_name.replace(" ", "_").lower()

best_model.write() \
    .overwrite() \
    .save(f"{BASE_PATH}/tests/{safe_model_name}")

if "cv_model" in globals():
    cv_model.bestModel.write() \
        .overwrite() \
        .save(f"{BASE_PATH}/tests/tuned_best_model")

test_data = test_df.select("features", "fare_amount")

test_predictions = best_model.transform(test_data)

test_with_residuals = test_predictions.withColumn(
    "residual",
    F.col("fare_amount") - F.col("prediction")
)

(
    test_with_residuals
        .select("prediction", "fare_amount", "residual")
        .coalesce(1)
        .write
        .mode("overwrite")
        .option("header", "true")
        .csv(f"{BASE_PATH}/tableau/final_predictions")
)

print(f"Best model saved at:  {BASE_PATH}/tests/{safe_model_name}")

if "cv_model" in globals():
    print(f"Tuned model saved at: {BASE_PATH}/tests/tuned_best_model")

print(f"Tableau export saved at: {BASE_PATH}/tableau/final_predictions")
''
print("\nBEST MODEL PERFORMANCE:")

if best_model_name in test_results:
    scores = test_results[best_model_name]
    print(
        f"{best_model_name} | "
        f"RMSE = ${scores['Test_RMSE']:.2f}, "
        f"R² = {scores['Test_R2']:.4f}"
    )


SAVING MODELS & EXPORTING RESULTS
Best model saved at:  /content/drive/MyDrive/NYC_Taxi_Yellow_2012/tests/1._linear_regression
Tuned model saved at: /content/drive/MyDrive/NYC_Taxi_Yellow_2012/tests/tuned_best_model
Tableau export saved at: /content/drive/MyDrive/NYC_Taxi_Yellow_2012/tableau/final_predictions

BEST MODEL PERFORMANCE:
1. Linear Regression | RMSE = $4.20, R² = 0.8101


In [12]:
train_df = spark.read.parquet(
    "/content/drive/MyDrive/NYC_Taxi_Yellow_2012/data/final/train_2012"
)

train_df.show(5)
train_df.printSchema()

+--------------------+-----------+
|            features|fare_amount|
+--------------------+-----------+
|[0.23388725290758...|       18.1|
|[-0.5203900857158...|        7.7|
|[2.49671926877781...|        4.9|
|[-0.5203900857158...|        9.3|
|[0.23388725290758...|        3.3|
+--------------------+-----------+
only showing top 5 rows
root
 |-- features: vector (nullable = true)
 |-- fare_amount: double (nullable = true)



In [13]:
import pandas as pd
import os
from pyspark.ml.regression import LinearRegressionModel, RandomForestRegressionModel, GBTRegressionModel



BASE_PATH = "/content/drive/MyDrive/NYC_Taxi_Yellow_2012"



if 'trained_models' in locals() and 'best_model_name' in locals():
    best_model = trained_models[best_model_name]
else:
    raise NameError(
        "Required variables 'trained_models' or 'best_model_name' are not defined.\n" +
        "Please ensure preceding cells (especially 'cBwEPgXyDvNo' and 'HoX2YZbYEDs_') have been executed."
    )



importances_array = []
if isinstance(best_model, (RandomForestRegressionModel, GBTRegressionModel)):
    if hasattr(best_model, "featureImportances"):
        importances_array = best_model.featureImportances.toArray()
    else:
        print(f"Warning: Model {type(best_model).__name__} does not have 'featureImportances'.")
elif isinstance(best_model, LinearRegressionModel):
    if hasattr(best_model, "coefficients"):
        importances_array = best_model.coefficients.toArray()
    else:
        print(f"Warning: Model {type(best_model).__name__} does not have 'coefficients'.")
else:
    print(f"Warning: Model type {type(best_model)} is not supported for feature importance/coefficient extraction.")



numeric_features_names = [
    "passenger_count",
    "trip_distance",
    "congestion_surcharge",
    "pickup_hour",
    "is_peak_hour",
    "is_weekend"
]
one_hot_encoded_dow_names = [f"dow_vector_day_{i}" for i in range(7)]

feature_names = numeric_features_names + one_hot_encoded_dow_names



if len(feature_names) != len(importances_array):
    print(f"Warning: Mismatch between reconstructed feature names count ({len(feature_names)}) and actual importances count ({len(importances_array)}). Using generic names as fallback.")

    feature_names = [f"feature_{i}" for i in range(len(importances_array))]


fi_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances_array
})

fi_df = fi_df.sort_values("Importance", ascending=False)


os.makedirs(f"{BASE_PATH}/tableau", exist_ok=True)
fi_df.to_csv(f"{BASE_PATH}/tableau/feature_importance.csv", index=False)

print(f"Feature importance data saved to: {BASE_PATH}/tableau/feature_importance.csv")


Feature importance data saved to: /content/drive/MyDrive/NYC_Taxi_Yellow_2012/tableau/feature_importance.csv


In [14]:

import pandas as pd
import os
from pyspark.ml.regression import (
    LinearRegressionModel,
    GBTRegressionModel,
    RandomForestRegressionModel
)
from pyspark.sql import SparkSession
from pyspark.ml.evaluation import RegressionEvaluator
import re

spark = SparkSession.builder.getOrCreate()


BASE_PATH = "/content/drive/MyDrive/NYC_Taxi_Yellow_2012"


if 'best_model_name' not in locals():
    print("Warning: best_model_name not found in current scope. Defaulting to '2. Gradient Boosted Trees'.")
    best_model_name = "2. Gradient Boosted Trees"


if 'trained_models' in globals() and best_model_name in trained_models:
    best_model = trained_models[best_model_name]
else:
    print(f"Warning: 'trained_models' or '{best_model_name}' not found in current scope. Attempting to load best model from disk.")

    safe_model_name = best_model_name.replace(" ", "_").lower()
    model_path = f"{BASE_PATH}/tests/{safe_model_name}"

    try:

        if "linear regression" in safe_model_name:
            best_model = LinearRegressionModel.load(model_path)
        elif "gradient boosted trees" in safe_model_name:
            best_model = GBTRegressionModel.load(model_path)
        elif "random forest" in safe_model_name:
            best_model = RandomForestRegressionModel.load(model_path)
        else:
            raise ValueError(f"Unknown model type for '{best_model_name}'. Cannot load from disk.")

        print(f"Successfully loaded '{best_model_name}' from {model_path}")
    except Exception as e:
        raise RuntimeError(f"Could not load best_model from disk at {model_path}. Ensure previous cells were executed to train and save the model, or check the path. Error: {e}")


test_df = spark.read.parquet(f"{BASE_PATH}/data/final/test_2012")

evaluator_rmse = RegressionEvaluator(labelCol="fare_amount", metricName="rmse")
evaluator_mae = RegressionEvaluator(labelCol="fare_amount", metricName="mae")
evaluator_r2 = RegressionEvaluator(labelCol="fare_amount", metricName="r2")

predictions = best_model.transform(test_df)

rmse = evaluator_rmse.evaluate(predictions)
mae = evaluator_mae.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)

metrics_df = pd.DataFrame({
    "Model": [re.sub(r"^\d+\.\s*", "", best_model_name).strip()],
    "RMSE": [rmse],
    "MAE": [mae],
    "R2": [r2]
})

metrics_df.to_csv(f"{BASE_PATH}/tableau/model_comparison.csv", index=False)